# indice

- [1 Load libs](#1-Load-libs)
- [2 Config path](#2-Config-path)


## 1 Load libs

In [1]:
# Instalar PySpark e PyArrow se necessário
try:
    import pyspark
    import pyarrow
    print("PySpark e PyArrow já estão instalados.")
except ImportError:
    import subprocess
    import sys
    print("Instalando PySpark e PyArrow...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspark", "pyarrow"])
    print("Instalação concluída.")

# Configuração Otimizada do Spark
import os
import sys
from pyspark.sql import SparkSession

# Configurar variáveis para usar o Java incluído no PySpark
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

try:
    print("Tentando inicializar a sessão Spark com a correção de host...")
    spark = SparkSession.builder \
        .appName("AntiMoneyLaundering_Optimized") \
        .master("local[*]") \
        .config("spark.driver.memory", "20g") \
        .config("spark.driver.host", "127.0.0.1") \
        .config("spark.driver.maxResultSize", "4g") \
        .config("spark.sql.shuffle.partitions", "8") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .getOrCreate()
    
    print("\n✅ Sessão Spark iniciada com sucesso!")
    print(f"Versão do Spark: {spark.version}")
    
    
except Exception as e:
    print(f"\n❌ Erro ao inicializar Spark: {e}")
    print("\nVerifique se o Firewall do Windows não está bloqueando o Java ou Python.")
    spark = None

Instalando PySpark e PyArrow...
Instalação concluída.
Tentando inicializar a sessão Spark com a correção de host...
Instalação concluída.
Tentando inicializar a sessão Spark com a correção de host...

✅ Sessão Spark iniciada com sucesso!
Versão do Spark: 3.5.1

✅ Sessão Spark iniciada com sucesso!
Versão do Spark: 3.5.1


## 2 Config path

In [2]:
# path_data_raw = r'C:\Users\win\Desktop\TCC\money_laundering\data\raw'
# path_accounts_df = r"C:\Users\win\Desktop\TCC\money_laundering\data\external\LI-Medium_accounts.csv"
# path_trans_df = r"C:\Users\win\Desktop\TCC\money_laundering\data\external\LI-Medium_Trans.csv"

In [3]:
path_data_raw = r'C:\Users\win\Desktop\TCC\money_laundering\data\raw'
path_accounts_df = r"C:\Users\win\Desktop\TCC\money_laundering\data\external\LI-Large_accounts.csv"
path_trans_df = r"C:\Users\win\Desktop\TCC\money_laundering\data\external\LI-Large_Trans.csv"

## 3 Union datasets

In [4]:
# Verificar se Spark foi inicializado
import time

if spark is None:
    print("❌ Spark não foi inicializado. Usando pandas...")
    # Fallback para pandas
    import pandas as pd
    
    start_time = time.time()
    accounts_df = pd.read_csv(path_accounts_df)
    trans_df = pd.read_csv(path_trans_df, nrows=100000)  # Limitar linhas
    end_time = time.time()
    
    print(f"Carregado com pandas - Accounts: {accounts_df.shape}, Trans: {trans_df.shape}")
    print(f"⏱️ Tempo de carregamento: {end_time - start_time:.2f} segundos")
else:
    print("✅ Usando PySpark para carregar os dados...")
    
    # Medição do tempo total
    inicio_total = time.time()
    
    # Carregar arquivos com PySpark
    print("Carregando accounts_df...")
    start_time = time.time()
    accounts_df = spark.read.csv(path_accounts_df, header=True, inferSchema=True)
    accounts_count = accounts_df.count()
    end_time = time.time()
    print(f"  ⏱️ Accounts: {end_time - start_time:.2f}s ({accounts_count:,} linhas)")
    
    print("Carregando trans_df...")
    start_time = time.time()
    trans_df = spark.read.csv(path_trans_df, header=True, inferSchema=True)
    trans_count = trans_df.count()
    end_time = time.time()
    print(f"  ⏱️ Transactions: {end_time - start_time:.2f}s ({trans_count:,} linhas)")
    
    # Renomear colunas no trans_df
    start_time = time.time()
    old_columns = trans_df.columns
    new_columns = ['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 
                   'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 
                   'Payment Format', 'Is Laundering']
    
    for old_col, new_col in zip(old_columns, new_columns):
        trans_df = trans_df.withColumnRenamed(old_col, new_col)
    end_time = time.time()
    print(f"  ⏱️ Renomeação de colunas: {end_time - start_time:.2f}s")
    
    # 1. Juntar transações com informações da conta de origem (remetente)
    print("Realizando primeiro join...")
    start_time = time.time()
    trans_enriched_df = trans_df.join(
        accounts_df,
        (trans_df['From Bank'] == accounts_df['Bank ID']) & 
        (trans_df['From Account'] == accounts_df['Account Number']),
        'left'
    )
    
    # Renomear colunas para evitar conflitos
    trans_enriched_df = trans_enriched_df \
        .withColumnRenamed('Bank Name', 'From Bank Name') \
        .withColumnRenamed('Entity ID', 'From Entity ID') \
        .withColumnRenamed('Entity Name', 'From Entity Name')
    end_time = time.time()
    print(f"  ⏱️ Primeiro join: {end_time - start_time:.2f}s")
    
    # 2. Juntar o resultado com informações da conta de destino (destinatário)
    print("Realizando segundo join...")
    start_time = time.time()
    accounts_df_to = accounts_df.select(
        accounts_df['Bank ID'].alias('To_Bank_ID'),
        accounts_df['Account Number'].alias('To_Account_Number'),
        accounts_df['Bank Name'].alias('To Bank Name'),
        accounts_df['Entity ID'].alias('To Entity ID'),
        accounts_df['Entity Name'].alias('To Entity Name')
    )
    
    trans_enriched_df = trans_enriched_df.join(
        accounts_df_to,
        (trans_enriched_df['To Bank'] == accounts_df_to['To_Bank_ID']) & 
        (trans_enriched_df['To Account'] == accounts_df_to['To_Account_Number']),
        'left'
    )
    end_time = time.time()
    print(f"  ⏱️ Segundo join: {end_time - start_time:.2f}s")
    
    # Contagem final
    start_time = time.time()
    final_count = trans_enriched_df.count()
    end_time = time.time()
    print(f"  ⏱️ Contagem final: {end_time - start_time:.2f}s")
    
    # Tempo total
    fim_total = time.time()
    tempo_total = fim_total - inicio_total
    
    # Exibir resultado
    print("✅ Tabela de Transações Enriquecida criada com sucesso!")
    print(f"📊 Total de linhas: {final_count:,}")
    print(f"⏱️ TEMPO TOTAL DE PROCESSAMENTO: {tempo_total:.2f} segundos")
    print(f"⚡ Performance: {final_count/tempo_total:,.0f} registros/seg")
    
    print("\nPrimeiras 5 linhas:")
    trans_enriched_df.show(5, truncate=False)

✅ Usando PySpark para carregar os dados...
Carregando accounts_df...
  ⏱️ Accounts: 5.37s (2,079,627 linhas)
Carregando trans_df...
  ⏱️ Accounts: 5.37s (2,079,627 linhas)
Carregando trans_df...
  ⏱️ Transactions: 42.40s (176,066,557 linhas)
  ⏱️ Renomeação de colunas: 0.06s
Realizando primeiro join...
  ⏱️ Primeiro join: 0.05s
Realizando segundo join...
  ⏱️ Segundo join: 0.03s
  ⏱️ Transactions: 42.40s (176,066,557 linhas)
  ⏱️ Renomeação de colunas: 0.06s
Realizando primeiro join...
  ⏱️ Primeiro join: 0.05s
Realizando segundo join...
  ⏱️ Segundo join: 0.03s
  ⏱️ Contagem final: 102.99s
✅ Tabela de Transações Enriquecida criada com sucesso!
📊 Total de linhas: 176,066,557
⏱️ TEMPO TOTAL DE PROCESSAMENTO: 150.91 segundos
⚡ Performance: 1,166,695 registros/seg

Primeiras 5 linhas:
  ⏱️ Contagem final: 102.99s
✅ Tabela de Transações Enriquecida criada com sucesso!
📊 Total de linhas: 176,066,557
⏱️ TEMPO TOTAL DE PROCESSAMENTO: 150.91 segundos
⚡ Performance: 1,166,695 registros/seg

Pri

In [5]:
# Importar funções necessárias do PySpark
from pyspark.sql.functions import min as spark_min, max as spark_max, col, collect_set

# Encontrar máximo e mínimo da coluna Timestamp
timestamp_stats = trans_enriched_df.agg(
    spark_min(col("Timestamp")).alias("min_timestamp"),
    spark_max(col("Timestamp")).alias("max_timestamp")
).collect()[0]

timestamp_min = timestamp_stats["min_timestamp"]
timestamp_max = timestamp_stats["max_timestamp"]

print(f"Timestamp mínima: {timestamp_min}")
print(f"Timestamp máxima: {timestamp_max}")

Timestamp mínima: 2022/08/01 00:00
Timestamp máxima: 2023/01/12 10:18


## 3 Salve data raw

In [6]:
# # Salvar o DataFrame resultante em um novo arquivo CSV
# trans_enriched_df.to_csv(r'C:\Users\win\Desktop\TCC\money_laundering\data\raw\trans_enriched.csv', index=False)


In [7]:
# trans_enriched_df.head()

In [8]:
# Verificar se temos dados carregados e qual tipo
if 'trans_enriched_df' in locals():
    if spark is not None and hasattr(trans_enriched_df, 'agg'):
        # Usando PySpark DataFrame
        print("✅ Analisando dados com PySpark...")
        
        # Importar funções necessárias do PySpark
        from pyspark.sql.functions import min as spark_min, max as spark_max, col, collect_set
        
        # Encontrar máximo e mínimo da coluna Timestamp
        timestamp_stats = trans_enriched_df.agg(
            spark_min(col("Timestamp")).alias("min_timestamp"),
            spark_max(col("Timestamp")).alias("max_timestamp")
        ).collect()[0]
        
        timestamp_min = timestamp_stats["min_timestamp"]
        timestamp_max = timestamp_stats["max_timestamp"]
        
        print(f"Timestamp mínima: {timestamp_min}")
        print(f"Timestamp máxima: {timestamp_max}")
        
        # Mostrar valores únicos (limitando para performance)
        print(f"\nPrimeiros 20 valores únicos de Timestamp:")
        unique_timestamps = trans_enriched_df.select("Timestamp").distinct().limit(20).collect()
        for row in sorted(unique_timestamps, key=lambda x: x["Timestamp"]):
            print(row["Timestamp"])
            
        print(f"\nTotal de timestamps únicos: {trans_enriched_df.select('Timestamp').distinct().count()}")
        
    else:
        # Usando pandas DataFrame
        print("✅ Analisando dados com pandas...")
        timestamp_min = trans_enriched_df['Timestamp'].min()
        timestamp_max = trans_enriched_df['Timestamp'].max()
        
        print(f"Timestamp mínima: {timestamp_min}")
        print(f"Timestamp máxima: {timestamp_max}")
        
        # Mostrar valores únicos ordenados para melhor visualização
        print(f"\nTodas as Timestamps presentes no dataset:")
        unique_vals = sorted(trans_enriched_df['Timestamp'].unique())
        print(unique_vals[:20])  # Mostrar apenas os primeiros 20
        if len(unique_vals) > 20:
            print(f"... e mais {len(unique_vals) - 20} valores")
else:
    print("❌ Dados não foram carregados ainda. Execute as células anteriores primeiro.")

✅ Analisando dados com PySpark...
Timestamp mínima: 2022/08/01 00:00
Timestamp máxima: 2023/01/12 10:18

Primeiros 20 valores únicos de Timestamp:
Timestamp mínima: 2022/08/01 00:00
Timestamp máxima: 2023/01/12 10:18

Primeiros 20 valores únicos de Timestamp:
2022/08/01 00:00
2022/08/01 00:02
2022/08/01 00:03
2022/08/01 00:07
2022/08/01 00:08
2022/08/01 00:10
2022/08/01 00:14
2022/08/01 00:18
2022/08/01 00:26
2022/08/01 00:27
2022/08/01 00:30
2022/08/01 00:38
2022/08/01 00:44
2022/08/01 00:45
2022/08/01 00:46
2022/08/01 00:48
2022/08/01 00:52
2022/08/01 00:58
2022/08/01 01:15
2022/08/01 01:27
2022/08/01 00:00
2022/08/01 00:02
2022/08/01 00:03
2022/08/01 00:07
2022/08/01 00:08
2022/08/01 00:10
2022/08/01 00:14
2022/08/01 00:18
2022/08/01 00:26
2022/08/01 00:27
2022/08/01 00:30
2022/08/01 00:38
2022/08/01 00:44
2022/08/01 00:45
2022/08/01 00:46
2022/08/01 00:48
2022/08/01 00:52
2022/08/01 00:58
2022/08/01 01:15
2022/08/01 01:27

Total de timestamps únicos: 143184

Total de timestamps úni